# Comparing WALS and Grambank with pandas

Öykü Yaylalı

## Discussion (about 300 words)

### What I learnt

*To write.*

### Difficulties and how I overcame them

*To write.*

### Suggestions for improvement

*To write.*

# Part 1: Loading and auditing the data

## Question 1: Loading and auditing

In [77]:
#import pandas
import pandas as pd 

#reads the csv file and turns it into a dataframe to be able to process it with python
wals_languages = pd.read_csv("data/wals/cldf/languages.csv")

#isna asks if the value is missing and sum counts the number of missing values
wals_languages["ID"].isna().sum()
wals_languages[wals_languages["ID"].isna()]

#only empty cells count as missing
wals_languages = pd.read_csv(
    "data/wals/cldf/languages.csv",
    keep_default_na=False,
    na_values=[""],
)

# checked it again 
print(wals_languages["ID"].isna().sum())

# load the values table the same way
wals_values = pd.read_csv(
    "data/wals/cldf/values.csv",
    keep_default_na=False,
    na_values=[""],
)

# count Nandi's data points
print(len(wals_values[wals_values["Language_ID"] == "nan"]))

0
74


In [78]:
wals_languages.nunique()

# column types for the WALS languages table
wals_languages_dtypes = {
    "Macroarea": "category",
    "Family": "category",
    "Subfamily": "category",
    "Samples_100": "boolean",
    "Samples_200": "boolean",
}

wals_languages = pd.read_csv(
    "data/wals/cldf/languages.csv",
    dtype=wals_languages_dtypes,
    keep_default_na=False,
    na_values=[""],
)

wals_languages.info()

wals_parameters = pd.read_csv(
    "data/wals/cldf/parameters.csv",
    keep_default_na=False,
    na_values=[""],
)

wals_parameters.info()
wals_parameters.nunique()

<class 'pandas.DataFrame'>
RangeIndex: 3573 entries, 0 to 3572
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   ID            3573 non-null   str     
 1   Name          3573 non-null   str     
 2   Macroarea     2662 non-null   category
 3   Latitude      2662 non-null   float64 
 4   Longitude     2662 non-null   float64 
 5   Glottocode    2645 non-null   str     
 6   ISO639P3code  2614 non-null   str     
 7   Family        2662 non-null   category
 8   Subfamily     911 non-null    category
 9   Genus         2662 non-null   str     
 10  GenusIcon     625 non-null    str     
 11  ISO_codes     2640 non-null   str     
 12  Samples_100   2662 non-null   boolean 
 13  Samples_200   2662 non-null   boolean 
 14  Country_ID    2655 non-null   str     
 15  Source        2499 non-null   str     
 16  Parent_ID     3319 non-null   str     
dtypes: boolean(2), category(3), float64(2), str(10)
memory usage: 6

ID             192
Name           192
Description      0
ColumnSpec       0
Chapter_ID     144
dtype: int64

In [79]:
wals_values = pd.read_csv(
    "data/wals/cldf/values.csv",
    dtype={
        "Language_ID": "category",
        "Parameter_ID": "category",
        "Code_ID": "category",
    },
    keep_default_na=False,
    na_values=[""],
)

In [80]:
wals_values.nunique()

ID              76475
Language_ID      2660
Parameter_ID      192
Value              28
Code_ID          1139
Comment          2068
Source          29715
Example_ID       1444
dtype: int64

In [81]:
# remaining WALS table
wals_codes = pd.read_csv(
    "data/wals/cldf/codes.csv",
    dtype={"Parameter_ID": "category"},
    keep_default_na=False,
    na_values=[""],
)

# Grambank tables
gb_languages = pd.read_csv(
    "data/grambank/cldf/languages.csv",
    dtype={
        "Macroarea": "category",
        "Family_name": "category",
        "level": "category",
    },
    keep_default_na=False,
    na_values=[""],
)

gb_parameters = pd.read_csv(
    "data/grambank/cldf/parameters.csv",
    keep_default_na=False,
    na_values=[""],
)

gb_codes = pd.read_csv(
    "data/grambank/cldf/codes.csv",
    dtype={"Parameter_ID": "category"},
    keep_default_na=False,
    na_values=[""],
)

# in Grambank, ? means unknown
gb_values = pd.read_csv(
    "data/grambank/cldf/values.csv",
    dtype={
        "Language_ID": "category",
        "Parameter_ID": "category",
        "Code_ID": "category",
        "Value": "Int8",
    },
    keep_default_na=False,
    na_values=["", "?"],
)

In [82]:
# every ID must be filled and unique
print(wals_languages["ID"].isna().sum(), wals_languages["ID"].is_unique)
print(wals_parameters["ID"].isna().sum(), wals_parameters["ID"].is_unique)
print(wals_codes["ID"].isna().sum(), wals_codes["ID"].is_unique)
print(wals_values["ID"].isna().sum(), wals_values["ID"].is_unique)
print(gb_languages["ID"].isna().sum(), gb_languages["ID"].is_unique)
print(gb_parameters["ID"].isna().sum(), gb_parameters["ID"].is_unique)
print(gb_codes["ID"].isna().sum(), gb_codes["ID"].is_unique)
print(gb_values["ID"].isna().sum(), gb_values["ID"].is_unique)

# links in the values tables must be filled (they repeat, so not unique)
print(wals_values["Language_ID"].isna().sum(), wals_values["Parameter_ID"].isna().sum())
print(gb_values["Language_ID"].isna().sum(), gb_values["Parameter_ID"].isna().sum())

# Nandi's data points, still 74 after adding the column types
print(len(wals_values[wals_values["Language_ID"] == "nan"]))

# share of ? in Grambank
print(gb_values["Value"].isna().mean())

0 True
0 True
0 True
0 True
0 True
0 True
0 True
0 True
0 0
0 0
74
0.1803139497761868


**Reflect (Q1).** 
Loading the data looked simple, but pandas silently turned Nandi’s ID nan into a missing value. Without keep_default_na=False, I would have lost its 74 data points with no error message. After the fix, all 8 ID columns have 0 missing values and are unique, the link columns have 0 missing values in both databases, and Nandi still has 74 rows after changing the column types. In Grambank, 18.0% of the values are ?, so almost one in five cells is unknown, which I need to remember when comparing with WALS. The lesson for me is not to trust the default settings and to check the data right after loading it.


## Question 2: WALS hierarchy

**Predict (Q2).** 
I think the hierarchy goes from the biggest level on top: genus > family > subfamily > language. So I expect these parents:

Language: its parent is the subfamily if it has one, and if it doesn’t have a subfamily, its parent is the family.

Subfamily: its parent is the family.

Family: its parent is the genus.

Genus: it is at the very top, so I don’t expect it to have a parent.

My reasons: some languages don’t have a subfamily at all, so they must connect directly to the family. I also don’t think it can be family > genus > subfamily, because family and subfamily feels like they should be stacked on top of each other and not separated by something in between. And in Parent_ID the values are coded with genera, which made me think genus is the biggest parent.

In [83]:
wals_languages["Type"] = "language"
wals_languages.loc[wals_languages["ID"].str.startswith("genus-"), "Type"] = "genus"
wals_languages.loc[wals_languages["ID"].str.startswith("family-"), "Type"] = "family"
wals_languages.loc[wals_languages["ID"].str.startswith("subfamily-"), "Type"] = "subfamily"

print(wals_languages["Type"].value_counts())

Type
language     2662
genus         625
family        254
subfamily      32
Name: count, dtype: int64


In [84]:
wals_parents = wals_languages[["ID", "Type"]].rename(
    columns={"ID": "Parent_ID", "Type": "Parent_Type"}
)

wals_hierarchy = wals_languages.merge(wals_parents, on="Parent_ID", how="left")
print(len(wals_languages), len(wals_hierarchy))
print((wals_hierarchy["Parent_ID"].notna() & wals_hierarchy["Parent_Type"].isna()).sum())

pd.crosstab(wals_hierarchy["Type"], wals_hierarchy["Parent_Type"].fillna("none"))


3573 3573
0


Parent_Type,family,genus,none,subfamily
Type,,,,
family,0,0,254,0
genus,462,0,0,163
language,0,2662,0,0
subfamily,32,0,0,0


**What I got wrong and why (Q2).** 
I got it wrong because I saw that the Parent_ID of the languages was always a genus, and I thought this meant genus is the biggest parent. But it only means that the genus is directly above the language. 

If I had scrolled down to the bottom of the table, I could have understood this much earlier, because the group rows are there, and I could see the whole hierarchy with my own eyes before writing any code.

## Question 3: Grambank and the family column

In [85]:
#extract the rows that are languages from both datasets and make copies of them

gb_languages = gb_languages[gb_languages["level"] == "language"].copy()
wals_languages = wals_languages[wals_languages["Type"] == "language"].copy()
print(len(gb_languages), len(wals_languages))

#check how many languages are missing family information in both datasets

print(gb_languages["Family_name"].isna().sum())
print(wals_languages["Family"].isna().sum())

2393 2662
101
0


In [86]:
#create family in grambank
gb_languages["family"] = gb_languages["Family_name"].astype("object")
isolate = gb_languages["family"].isna()
gb_languages.loc[isolate, "family"] = "isolate-" + gb_languages.loc[isolate, "ID"]

# same for WALS
wals_languages["family"] = wals_languages["Family"]

In [87]:
print(gb_languages["family"].nunique(), wals_languages["family"].nunique())

for df in [gb_languages, wals_languages]:
    top5 = df["family"].value_counts().head(5)
    print(top5)
    print(top5.sum() / len(df))

315 254
family
Austronesian      520
Atlantic-Congo    311
Sino-Tibetan      185
Afro-Asiatic      119
Pama-Nyungan       81
Name: count, dtype: int64
0.5081487672377768
family
Austronesian     324
Niger-Congo      324
Indo-European    176
Sino-Tibetan     146
Afro-Asiatic     145
Name: count, dtype: int64
0.4188580015026296


**Reflect (Q3).**
I counted every isolate as its own family, so Grambank has 315 families (101 are isolates).

Another way is to match the datasets with Glottocode and remove the 65 languages that both datasets say are isolates, because I want to compare the two datasets and isolates have no family to compare with. Then Grambank would have 250 families and the top five share would change i suppose. But im not sure what effect would it have on the actual study for now.

## Question 4: Coders

In [88]:
#explode the coders column into separate rows

gb_values["coder_list"] = gb_values["Coders"].str.split(";")
coder_values = gb_values.explode("coder_list")
coder_values[["Coders", "coder_list", "Value"]].head()

# analyze coder statistics of ? 

coder_values["is_missing"] = coder_values["Value"].isna()
coder_stats = coder_values.groupby("coder_list")["is_missing"].agg(["size", "mean"])
coder_stats = coder_stats[coder_stats["size"] >= 1000].sort_values("mean")
print(len(coder_stats))

coder_stats



76


,size,mean
coder_list,,
MN,1170,0.005983
IC,1170,0.029060
TWE,4210,0.047031
LR,1434,0.050907
SVM,4093,0.061569
GF,2087,0.073790
MY,8262,0.079158
CK,1755,0.089459
RSI,14581,0.089980


**Reflect (Q4).**
The share of ? goes from 0.6% to 40% between coders. My personal explanation would be about the languages: they are small, less described or endangered languages. The languages explain a big part but not everything. I think blaming the coders is too harsh, because probably an expert worked on each of these languages. But then again, the languages explain a big part but not everything. 

## Question 5: Long and wide tables, coverage

In [89]:
# WALS: get the labels from codes.csv
wals_labels = wals_codes[["ID", "Name"]]
wals_labels = wals_labels.rename(columns={"ID": "Code_ID", "Name": "Value_label"})

# attach the labels to the values
wals_long = wals_values.merge(wals_labels, on="Code_ID", how="left")

# keep only language rows
is_language = wals_long["Language_ID"].isin(wals_languages["ID"])
wals_long = wals_long[is_language]

# keep only the four columns we need
columns_to_keep = ["Language_ID", "Parameter_ID", "Value", "Value_label"]
wals_long = wals_long[columns_to_keep].copy()


# Grambank: same steps (label is the Description column: absent / present)
gb_labels = gb_codes[["ID", "Description"]]
gb_labels = gb_labels.rename(columns={"ID": "Code_ID", "Description": "Value_label"})

gb_long = gb_values.merge(gb_labels, on="Code_ID", how="left")

is_language = gb_long["Language_ID"].isin(gb_languages["ID"])
gb_long = gb_long[is_language]

gb_long = gb_long[columns_to_keep].copy()

print(wals_long.shape, gb_long.shape)

(76475, 4) (428872, 4)


**Reflect (Q5).**
No language in WALS is documented for all 192 features, so dropping languages with missing values would leave us with nothing: the median language has only 19 features filled, and only about 15% of the whole table is filled. But missing data is not always the worst case, because we can still use what is known about a language for other features, so for each analysis I would only keep the languages that have the features I need. Only 401 languages have all of the twenty best-covered features, almost half come from five big families, most are in Eurasia, Africa and Papunesia, and only 10 are Australian. Using only the complete cases would bias the results towards these languages.

# Part 2: Linking the two databases

## Question 6: Languages without a Glottocode

In [90]:
n_glottocode = wals_languages[wals_languages["Glottocode"].isna()]



In [91]:
print(wals_languages["ISO639P3code"].notna().sum(), len(wals_languages))
print(gb_languages["ISO639P3code"].notna().sum(), len(gb_languages))
print(n_glottocode["ISO639P3code"].notna().sum())

2614 2662
0 2393
11


**Reflect (Q6).**
17 WALS languages have no Glottocode, and five of them list two ISO codes (e.g. Azerbaijani: azb azj, Arrernte: aer are). My guess is that WALS merged close varieties into one entry, while Glottolog and Grambank keep the varieties apart (Grambank has North Azerbaijani and Eastern Arrernte), so the merged entry has no single Glottocode to point to. Tasmanian is the case I don’t fully get: it has no ISO code in WALS, and Glottolog gives Tasmanian both a general Glottocode and codes for its subtypes, so I don’t know why WALS has none.

## Question 7: Doculects and deduplication

In [94]:
# only languages that have a Glottocode 
with_code = wals_languages[wals_languages["Glottocode"].notna()]

# how many doculects does each Glottocode have?
group_size = with_code.groupby("Glottocode").size()

# how many Glottocodes have 1, 2, 3, ... rows
print(group_size.value_counts().sort_index())

# keep only the Glottocodes that appear more than once
dup_codes = group_size[group_size > 1].index
dup_langs = with_code[with_code["Glottocode"].isin(dup_codes)]
print(len(dup_codes), len(dup_langs))

1     2409
2       69
3       13
4        6
6        4
11       1
Name: count, dtype: int64
93 236


**Reflect (Q7).** *Three to six lines, quoting at least one of my own results.*

## Question 8: Outer merge

**Predict (Q8).** *Write before running the code below. Do not edit afterwards.*

**What I got wrong and why (Q8).** *To write after the result.*

**Reflect (Q8).** *Three to six lines, quoting at least one of my own results.*

## Question 9: Metadata comparison

**Reflect (Q9).** *Three to six lines, quoting at least one of my own results.*

## Question 10: Genealogical classification

**Reflect (Q10).** *Three to six lines, quoting at least one of my own results.*

# Part 3: Harmonising features

## Question 11: Mapping table

**Reflect (Q11).** *Three to six lines, quoting at least one of my own results.*

## Question 12: Word order as a single variable

**Reflect (Q12).** *Three to six lines, quoting at least one of my own results.*

## Question 13: Cohen's kappa

**Predict (Q13).** *Write before running the code below. Do not edit afterwards.*

**What I got wrong and why (Q13).** *To write after the result.*

**Reflect (Q13).** *Three to six lines, quoting at least one of my own results.*

## Question 14: Disagreement analysis

**Reflect (Q14).** *Three to six lines, quoting at least one of my own results.*

## Question 15: Contradiction tests

**Reflect (Q15).** *Three to six lines, quoting at least one of my own results.*

## Question 16: Combined verb-final variable

**Reflect (Q16).** *Three to six lines, quoting at least one of my own results.*

# Part 4: Typological studies

## Question 17: Distribution of 81A

**Reflect (Q17).** *Three to six lines, quoting at least one of my own results.*

## Question 18: Galton's problem

**Predict (Q18).** *Write before running the code below. Do not edit afterwards.*

**What I got wrong and why (Q18).** *To write after the result.*

**Reflect (Q18).** *Three to six lines, quoting at least one of my own results.*

## Question 19: Greenberg's Universal 4

**Reflect (Q19).** *Three to six lines, quoting at least one of my own results.*

## Question 20: Dryer (1992)

**Predict (Q20).** *Write before running the code below. Do not edit afterwards.*

**What I got wrong and why (Q20).** *To write after the result.*

**Reflect (Q20).** *Three to six lines, quoting at least one of my own results.*

## Question 21: Implication mining

**By hand (Q21).** *Paper computation before coding.*

**Reflect (Q21).** *Three to six lines, quoting at least one of my own results.*

## Question 22: Tone and latitude

**Reflect (Q22).** *Three to six lines, quoting at least one of my own results.*

## Question 23: Consonant and vowel inventories

**Reflect (Q23).** *Three to six lines, quoting at least one of my own results.*

## Question 24: Typological distance

**Reflect (Q24).** *Three to six lines, quoting at least one of my own results.*

## Question 25: Areality

**Reflect (Q25).** *Three to six lines, quoting at least one of my own results.*

# Part 5: Wrapping up

## Question 26: Pipeline and Parquet

**Reflect (Q26).** *Three to six lines, quoting at least one of my own results.*

## Question 27: Critical reading

*About half a page. To write.*